# PySpark intro

PySpark is the Python API to Apache Spark


a short note on notebooks in databricks - can choose between
- SQL notebook
- Python notebook

the choice makes all cells default to that choice eg SQL -> cells become SQL by default

In [0]:
DATA_PATH = "/Volumes/data/olympic_games/raw_data"

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True)

df_athletes

In [0]:
df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

In [0]:
(df_athletes.count(), len(df_athletes.columns))

## Infer schema

we note that age, height becomes strings thats because it has null values

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, inferSchema=True)
df_athletes

In [0]:
df_athletes.display()

## Define explicit schema


In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

## EDA

### PySpark transformations

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE', 'NOR', 'FIN', 'DEN', 'ISL') AND Medal != 'NA'"
).sort("NOC", "Medal").display()

## Spark SQL

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

spark.sql("""
          SELECT 
            Name,
            Sport,
            Event,
            Year,
            Medal
        FROM df_athletes_schema
        WHERE 
            NOC = 'SWE' AND 
            Sport = 'Table Tennis' AND 
            Medal != 'NA'
        ORDER BY Year;"""
).display()

## Count medals and plot

In [0]:
df_swe_medals = spark.sql("""
          SELECT
            sport,
            COUNT(medal) AS medal_count
          FROM df_athletes_schema
          WHERE noc = 'SWE' AND medal in ('Bronze', 'Silver', 'Gold')
          GROUP BY sport
          ORDER BY medal_count DESC
          """)

df_swe_medals.display()

In [0]:
fig = df_swe_medals.plot(kind='bar', x='medal_count', y='sport')
fig.update_layout(yaxis = {"autorange": 'reversed'})

## Ingesting data to Unity Catalog

In [0]:
%sql
SHOW SCHEMAS IN data;

CREATE TABLE IF NOT EXISTS data.olympic_games.sweden_medals as (
    SELECT
        name,
        age,
        height,
        weight,
        sport,
        year,
        medal
    FROM df_athletes_schema
    WHERE noc = 'SWE' AND medal in ('Bronze', 'Silver', 'Gold')
);

In [0]:
%sql
SELECT
    name,
    sport,
    year,
    medal
FROM data.olympic_games.sweden_medals
WHERE medal = 'Gold'
ORDER BY year ASC;